# Data Categorisation

This notebook documents how enrollment data from Australia, the United Kingdom, and New Zealand were mapped to a shared 11-category taxonomy, and how Australian government funding cluster assignments link to those same categories.

The same **CategoryKey 1–11** is applied across all datasets to ensure comparability in the regression analysis.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'data').exists():
    ROOT = ROOT.parent

FUND_PATH = ROOT / 'data' / 'clean' / 'AnnualFundingAUS2019-2026_with_category_key.csv'
print('Root:', ROOT)

Root: C:\Users\neddp\ECC3479-Project-JRGS


## 1. Category–Subject Crosswalk

Each country uses a different subject classification system. All three were crosswalked to the same 11 categories:

- **Australia** uses ASCED Field of Education (FOE) broad group codes (1–11). CategoryKeys correspond directly to FOE broad groups.
- **UK** uses HESA subject areas. Disciplines with 2016–2024 coverage use JACS codes (2016–2018) and CAH codes (2019–2024); disciplines with 2019–2024 coverage only use CAH codes.
- **New Zealand** uses the TEC Field of Study classification, which closely mirrors the ASCED structure and maps directly to the same category keys.

| Key | Category | Subject coverage | UK (HESA) | NZ (TEC) |
|:---:|----------|-----------------|-----------|----------|
| 1 | Natural & Physical Science | Biology, chemistry, physics, mathematics, statistics | Biological sciences; Mathematical sciences; Physical sciences | Natural and Physical Sciences |
| 2 | Information Technology | Computing, software engineering, data management, networking | Computer science / Computing | Information Technology |
| 3 | Engineering & Related Tech | Mechanical, civil, electrical, chemical, aerospace engineering | Engineering & technology / Engineering and technology | Engineering and Related Technologies |
| 4 | Architecture & Building | Architecture, construction, urban and regional planning | Architecture, building & planning / Architecture, building and planning | Architecture and Building |
| 5 | Environment & Related | Environmental science, agriculture, earth sciences, geography | Agriculture, food and related studies; Geography, earth and environmental studies | Agriculture, Environmental and Related |
| 6 | Health | Medicine, nursing, allied health, dentistry, pharmacy, veterinary science | Medicine & dentistry; Subjects allied to medicine / Medicine and dentistry; Veterinary sciences | Health |
| 7 | Education | Teacher education, early childhood, educational administration | Education / Education and teaching | Education |
| 8 | Management & Commerce | Business, accounting, finance, marketing, human resources | Business & administrative studies / Business and management | Management and Commerce |
| 9 | Society & Culture | Social sciences, law, economics, psychology, languages, history, philosophy, media | Psychology; Social sciences; Law; Language and area studies; Historical and philosophical studies; Media and communications | Society and Culture |
| 10 | Creative Arts | Visual arts, design, music, film, performing arts | Creative arts & design / Design and creative and performing arts | Creative Arts |
| 11 | Others | Mixed-field programmes, hospitality, personal services, general studies | Combined and general studies | Food, Hospitality and Personal Services; Mixed Field Programmes |

> **UK note:** Where two HESA names are shown separated by `/`, the first is the JACS name (used for 2016–2018 data) and the second is the CAH equivalent (used from 2019). For keys 1, 5, 9, and 11, only CAH-period data (2019–2024) was available due to taxonomy incompatibility.

## 2. AUS Funding Cluster Linkage

Australian per-student funding is set by **Funding Cluster**, which determines the split between the student contribution (HECS-HELP) and the Commonwealth contribution per equivalent full-time student (EFTSL). Each ASCED Field of Education is assigned to a cluster; categories spanning multiple clusters contain sub-fields with different rates.

The Job-ready Graduates Package (JRG, effective 2021) reassigned many fields to different clusters. The table below shows cluster assignments and average student contributions before (2019–2020) and after (2021–2024) JRG.

In [2]:
df = pd.read_csv(FUND_PATH)

def cluster_list(data, key):
    vals = sorted(data[data['CategoryKey'] == key]['FundingCluster'].unique(),
                  key=lambda x: int(x.split()[-1]))
    return ', '.join(v.replace('Funding ', '') for v in vals)

pre  = df[df['Year'].isin([2019, 2020])]
post = df[df['Year'].isin([2021, 2022, 2023, 2024])]

rows = []
for key in sorted(df['CategoryKey'].unique()):
    name     = df[df['CategoryKey'] == key]['Category'].iloc[0]
    pre_cl   = cluster_list(pre, key)
    post_cl  = cluster_list(post, key)
    pre_stu  = round(pre[pre['CategoryKey'] == key]['MaximumStudentContribution'].mean())
    post_stu = round(post[post['CategoryKey'] == key]['MaximumStudentContribution'].mean())
    chg      = round((post_stu / pre_stu - 1) * 100, 1)
    chg_str  = f'+{chg}%' if chg >= 0 else f'{chg}%'
    rows.append({
        'Key':                 key,
        'Category':            name,
        'Pre-JRG cluster(s)':  pre_cl,
        'Post-JRG cluster(s)': post_cl,
        'Student $ (pre)':     f'${pre_stu:,}',
        'Student $ (post)':    f'${post_stu:,}',
        '% change':            chg_str,
    })

display(pd.DataFrame(rows).set_index('Key'))

,Category,Pre-JRG cluster(s),Post-JRG cluster(s),Student $ (pre),Student $ (post),% change
Key,,,,,,
1,Natural & Physical Science,"Cluster 3, Cluster 7, Cluster 8","Cluster 2, Cluster 3","$9,487","$7,853",-17.2%
2,Information Technology,Cluster 3,Cluster 2,"$9,443","$8,305",-12.1%
3,Engineering & Related Tech,Cluster 7,Cluster 3,"$9,443","$8,305",-12.1%
4,Architecture & Building,Cluster 3,Cluster 2,"$9,443","$8,305",-12.1%
5,Environment & Related,Cluster 8,"Cluster 3, Cluster 4","$9,443","$5,446",-42.3%
6,Health,"Cluster 3, Cluster 5, Cluster 6, Cluster 8","Cluster 1, Cluster 2, Cluster 3, Cluster 4","$9,462","$8,828",-6.7%
7,Education,Cluster 4,Cluster 2,"$6,625","$4,126",-37.7%
8,Management & Commerce,Cluster 1,Cluster 1,"$11,056","$15,149",+37.0%
9,Society & Culture,"Cluster 1, Cluster 2, Cluster 3, Cluster 5","Cluster 1, Cluster 2, Cluster 3","$7,364","$12,157",+65.1%


> **JRG cluster interpretation:** Priority fields under JRG (Education, IT, Engineering, Architecture, Environment) moved to lower-numbered clusters, reducing student contributions. Non-priority fields (Management & Commerce, Creative Arts, Society & Culture, Others) moved to higher-cost clusters, raising student fees substantially. Health spans a wide range of clusters because sub-fields such as clinical medicine, nursing, and allied health have always attracted sharply different funding rates.